# Making your server go Brrrrrrrr

## (Context variables)

# Context

![context](context.jpeg)

### What is a context

A context in english language means the circumstances around some event.

In programming a context is the variables/configurations around which this program is being executed.

For example, If I am spinning some kind of program the context around it is the environment variables and the global variables around that program as well as the system arguments.

### Thread Context

Now if you think about it inside a program you'd be spinning multiple threads, it's not necessary that you'd want the program context to be the same as the thread context.

Thread locals come to the resecue.


In [14]:
import threading
import time

thread_local = threading.local()

def fun3():
    print(f"Attachee to thread 1 {thread_local.value}")

def fun1():
    thread_local.value = "Thread 1"
    time.sleep(5)
    print(f"Thread local for thread 1 {thread_local.value}")
    fun3()

def fun2():
    thread_local.value = "Thread 2"
    time.sleep(1)
    print(f"Thread local for thread 2 {thread_local.value}")


thread1 = threading.Thread(target=fun1)
thread2 = threading.Thread(target=fun2)

thread1.start()
thread2.start()

thread1.join()
thread2.join()




Thread local for thread 2 Thread 2
Thread local for thread 1 Thread 1
Attachee to thread 1 Thread 1


# Thread Locals

What are thread locals? 

A thread local is a variable tied to the thread in which you're running the code.

So let's say that I have a some code where I wanna open a database transaction and call have that database transaction do some queries then commit it.

Ideally there are two ways to do that.

Way 1:
```python
def query1(db_conn, query):
    db_conn.execute(query)

def query2(db_conn, query):
    db_conn.execute(query)

def main():
    db_conn = start_transaction()
    query1(db_conn)
    query2(db_conn)
    db_conn.commit()
```

Now this is an easy example, but let's say that the functions aren't in the same file it makes it harder to keep track of the transaction.

If only we had some hand local on transaction level? oh but we do.

Way 2:
```python
import threading

db_conn = threading.local()

def query1(query):
    db_conn.execute(query)

def query2(query):
    db_conn.execute(query)

def main():
    db_conn = start_transaction()
    query1(db_conn)
    query2(db_conn)
    db_conn.commit()
```

It's because this main function can be a target of a thread that you can easily create a thread local around it and import that database connection from anywhere and use it.




# Fiber Context

Thread locals are boomers in the world of context let's talk about the genz of context locals which are Fiber contexts

### What are fiber contexts?

Instead of tying the context to a thread local why don't we do something a bit more crazy, let's tie it to the execution context instead of a thread.

What's an execution context?

The execution context is either a coroutine or a thread so whenever you're openning a coroutine from the a thread or opening a new thread the exectuion context changes

![fiber_context](fiber_context.png)

In [15]:
import contextvars
import asyncio
import threading

ctx = contextvars.ContextVar("ctx")

async def nested_func():
    print(f"CTX within nested func {ctx.get()}")
    ctx.set("nested func")

async def async_fun1():
    try:
        ctx.get()
    except:
        print("Execution context changed")
    ctx.set("Async Task 1")
    await asyncio.sleep(3)
    print("async_fun1: ", ctx.get())
    await nested_func()
    print(f"CTX after nested func {ctx.get()}")


async def async_fun2():
    try:
        ctx.get()
    except:
        print("Execution context changed")
    ctx.set("Async Task 2")
    await asyncio.sleep(1)
    print("async_fun2: ",ctx.get())

def run():
    try:
        ctx.get()
    except:
        print("Execution context changed")
    ctx.set("Task 3")
    print("new thread", ctx.get())
    
          
await asyncio.gather(async_fun1(), async_fun2())
t = threading.Thread(target=run)
t.start()
t.join()


Execution context changed
Execution context changed
async_fun2:  Async Task 2
async_fun1:  Async Task 1
CTX within nested func Async Task 1
CTX after nested func nested func
Execution context changed
new thread Task 3


# Reset Tokens

![reset](reset.jpeg)


Let's say that you wanna create multi-level contexts, ik you don't but let's just say you want to.

What should you do? Open a new thread? It shuold be a bit easier right? 

When setting a context the var gives you a token this token gives you the ability to reset the context to the previous context so in conecpt you can do the following:

```python
    token = ctx.set()
    run_new_ctx_function()
    ctx.reset(token)
```



In [16]:
import contextvars
ctx = contextvars.ContextVar("ctx")

token = ctx.set("ABC")
token2 = ctx.set("CDE")

print(ctx.get())
ctx.reset(token2)
print(ctx.get())


CDE
ABC


# Putting all together

Now ideally how we use these contexts is that we tie them to an http request were when someone calls your API you open up a context and inside that context you start a db transaction and close that transaction when closing the context.

Let's try to build some kind of utility that helps you achieve the above one by one


In [17]:
import contextvars
import dataclasses


@dataclasses.dataclass
class Context:
    current = contextvars.ContextVar("current")
    val = None

    def __init__(self, val=1):
        self.val = val

    def service(**kwargs):
        return Context(**kwargs)

    def __enter__(self):
        self._token = self.current.set(self)

    def __exit__(self, exc_type, exc_value, tb):
        self.current.reset(self._token)


ctx = Context.current

with Context.service():
    print(ctx.get().val)

    with Context.service(val=5):
        print(ctx.get().val)

    print(ctx.get().val)

1
5
1
